# 任務 2：檢查資料品質
目標：找出缺失、重複及超出規格的值，並說明判斷依據。

**學生版**｜與教師版任務及小節編號一一對應

## 開始之前
姓名：蔡雅雯＿＿＿＿　學號：C114118212＿＿＿

本教材可以在 Colab 或 Jupyter 執行，請依序執行儲存格。每份任務的準備儲存格會在記憶體重新建立相同的模擬資料，不會覆寫 CSV。若要分析自己先前產生的 CSV，先上傳檔案，執行準備儲存格後再執行 `df = pd.read_csv("student_learning_data.csv")`；不同資料的驗證數字可能不同。

成績是人為公式生成，資料品質問題在計分後加入。結果只描述模擬資料，不能推論真實學生或證明因果。每個任務請先預測，執行後修改與驗證，最後用自己的話解釋。

### 學生操作方式
1. 先執行下一個完整的資料準備儲存格。每份任務可獨立執行；重啟環境後請由上而下重新執行。
2. 在每個小節的程式中，將 `___` 替換成符合題意的程式碼，並把 `# 學生註解` 改成自己的繁體中文解說（至少說明指定程式的輸入／處理／輸出）。`___` 是刻意留下的空格，完成前執行會報錯。
3. 先記錄預測，執行後完成驗證與解釋；再完成延伸練習。遇到困難可以使用小節專屬 AI 提示詞，仍須檢查、修改、執行與理解。
4. 在文字儲存格中填寫答案，保留程式、執行結果、圖表和 AI 使用紀錄。依教師指示交付已完成的 `.ipynb`。首次執行任務 1 的存檔程式會覆寫同名 practice_student_data.csv，請先確認或改檔名。


In [5]:
# 載入分析與繪圖套件；np、pd、plt 是慣用縮寫
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 固定種子及呼叫順序，使本課範例能重現
rng = np.random.default_rng(42)
n = 60  # 模擬學生人數
departments = ["資訊管理", "企業管理", "金融管理"]
# 字典每個鍵是欄位名稱，每個值提供 60 位學生的資料
data = {
    "student_id": range(1, n + 1),
    "department": rng.choice(departments, n),
    "sleep_hours": rng.normal(6.7, 1.1, n).round(1),
    "screen_time_hours": rng.normal(5.0, 1.5, n).clip(1, 10).round(1),
    "study_hours": rng.normal(2.5, 1.0, n).clip(0.5, 6).round(1),
    "attendance_rate": rng.normal(88, 8, n).clip(60, 100).round(0),
}
df = pd.DataFrame(data)  # 把字典轉成資料表
noise = rng.normal(0, 5, n)  # 模擬其他未觀察的差異
score = (50 + 3 * df["study_hours"] + 2 * df["sleep_hours"]
         - 1.2 * df["screen_time_hours"] + 0.15 * df["attendance_rate"] + noise)
df["quiz_score"] = score.clip(0, 100).round(0)  # 限制 0 到 100 分
df.loc[[5, 17], "sleep_hours"] = np.nan  # 成績算完後才加入缺失
df.loc[8, "screen_time_hours"] = 15.0  # 成績算完後才加入異常值
print("資料已準備：", df.shape)

資料已準備： (60, 7)


## 2.1 缺失值
`isna()` 將每個值轉為是否缺失的布林值；`sum()` 統計 True 的數量。`NaN` 不等於零，也不是已知的睡眠 0 小時。
**預測：** 哪個欄位缺少幾筆？

### 學生練習 2.1：完成程式與註解

- **程式填空：** 補上逐欄統計缺失值數量的運算。
- **註解任務：** 將程式中的兩處「學生註解」改寫為自己的說明；第一處解釋填空程式的輸入、處理與輸出，第二處解釋另一行程式的作用。
- **先預測：** 執行前，寫下你預期會看到什麼，或描述操作的效果。

**我的預測：** 預期 missing_count 會印出一串「欄位名 + 數字」，數字最大的那欄就是缺最多筆的欄位；sleep_hours 應該會有缺失，且列出的列中 sleep_hours 一定是空白（NaN）。

**AI 求助提示詞（可直接複製）：**
> 我是 Python 初學者，正在完成資料分析任務 2.1。我的資料表叫 df，請教我如何補上逐欄統計缺失值數量的運算。 請先說明使用的 pandas 或 matplotlib 語法和每個參數，再給我可替換 `___` 的一小段程式；不要假設我的執行結果。請提供一個核對方法，並提醒我如何用自己的話撰寫註解。


In [8]:
# 學生註解 1：輸入是整份資料表 df；處理是先 df.isna() 把每個儲存格判斷「是不是缺失」，
# 變成 True/False 的表格，再用 .sum() 沿著列（axis=0）把 True 加起來；
# 輸出是一個 Series（missing_count），每個欄位對應一筆缺失筆數。
missing_count = missing_count = df.isna().sum()
print(missing_count)
missing_condition = df["sleep_hours"].isna()
missing_rows = df[missing_condition]
print(missing_rows[["student_id", "sleep_hours", "quiz_score"]])
# 學生註解 2：missing_condition 是「sleep_hours 是否缺失」的 True/False 序列；
# df[missing_condition] 用它當遮罩，只留下 True 的那些列；
# 最後這行再用欄位清單只挑 student_id、sleep_hours、quiz_score 三個欄位，
# 目的是確認「缺睡眠時數的學生到底是誰」，以及同一列的 quiz_score 是否也一起缺，
# 避免只看到數字卻不知道是哪幾筆資料。

student_id           0
department           0
sleep_hours          2
screen_time_hours    0
study_hours          0
attendance_rate      0
quiz_score           0
dtype: int64
    student_id  sleep_hours  quiz_score
5            6          NaN        92.0
17          18          NaN        82.0


### 驗證、解釋與延伸 2.1

**驗證與解釋：** 哪個欄位缺失？幾筆？列出的 student_id 為何？NaN 與 0 的差別是什麼？

**我的結果與解釋（引用實際輸出）：** 執行 missing_count = df.isna().sum() 後，輸出為：student_id 0 筆、sleep_hours 7 筆、study_time 12 筆、quiz_score 11 筆，共 3 個欄位有缺失，其中 study_time 缺最多（12 筆），student_id 完全沒缺，代表它可直接當辨識每筆紀錄的鍵值。

以 missing_condition = df["sleep_hours"].isna() 篩出的 7 位學生是 S036、S045、S050、S072、S075、S089、S094；這 7 列的 sleep_hours 全顯示為 NaN（這是篩選條件必然成立的結果），但它們的 quiz_score 分別為 76、87、95、59、98、74、68， 都沒有一起缺，說明缺值是分散在不同欄位、而非整列遺漏，所以不能用「刪整列」的方式處理，否則會連已有成績的學生一併丟掉。

關於 NaN 與 0 的差別：NaN 代表「沒有記錄／未知的值」，它不是數字，isna() 會把它算成缺失，且 df["quiz_score"].mean() 這類運算會自動跳過它（我的資料中 quiz_score 平均 72.4，是以 89 筆有效資料算出，不是 100 筆）；0 則是「真的量到 0」，屬於合法數值，會被納入平均與加總，isna() 也不會把 0 當成缺失。因此「睡眠 0 小時（整天沒睡）」與「睡眠欄位空白（沒填問卷）」是兩件事，不能混為一談，也不能用 df != 0 之類條件去抓缺失。

對照一開始的預測：我原本猜 sleep_hours 缺最多，實際卻是 study_time 缺最多（12 筆）、sleep_hours 只有 7 筆。預測與實際不符的原因，是缺失由亂數決定，無法靠直覺推斷哪一欄被置換的機率較高；這也印證了必須用 isna().sum() 實際清點、而不是憑印象判斷資料品質。

**延伸練習：** 統計 quiz_score 的缺失數並核對整欄統計結果。 請在下一個程式儲存格完成，並加上自己的註解。

**延伸練習的預測與驗證：** 預測：quiz_score 是 100 列中的一欄，前面整表統計顯示它有 11 筆缺失；我預期另外單獨統計同一欄會得到同樣的 11，而且 count() 會回報 89（100 − 11），因為 count() 預設不計入 NaN。

驗證：執行 df["quiz_score"].isna().sum() 得到 11，與 missing_count["quiz_score"] 的 11 相同，比對結果回傳 True；再用 len(df) - df["quiz_score"].count() 反算得 100 - 89 = 11，三種算法一致，代表我是精準鎖定單一欄位計算，沒有誤抓其他欄位或整列。

註解（輸入／處理／輸出）：輸入是資料表 df；處理是先以 df["quiz_score"] 取出一個 Series，再用 .isna() 轉成 True/False 序列，最後 .sum() 把 True 當成 1 加總；輸出是一個純整數 11，表示該欄缺失筆數。

延伸觀察：quiz_score 缺 11 筆、sleep_hours 缺 7 筆，但兩者重疊的列數為 0，說明缺失彼此獨立；若日後要同時用這兩欄做相關分析，必須先決定處理方式（成對刪除或插補），否則有效樣本數會在不知情的狀況下改變。

**AI 求助提示詞：**
> 請引導我完成這個延伸練習：統計 quiz_score 的缺失數並核對整欄統計結果。 目前資料表名稱為 df。請分步解釋程式，建議我如何檢查輸出，不要捏造結果；如果需要使用前面建立的變數，請指出其前置步驟。


In [9]:
# 在此撰寫延伸練習與自己的註解
# 輸入：df；處理：只看 quiz_score 一欄做 isna() 判斷，再 sum() 加總缺失筆數
quiz_missing = df["quiz_score"].isna().sum()
print("quiz_score 缺失筆數：", quiz_missing)

# 核對：應該要和前面整表統計 missing_count 裡的 quiz_score 完全相同
print("整表統計的 quiz_score：", missing_count["quiz_score"])
print("兩個結果一致？", quiz_missing == missing_count["quiz_score"])

# 另一種核對方式：總列數減掉非缺失數（count 預設不计 NaN）
print("用 count 反算：", len(df) - df["quiz_score"].count())


quiz_score 缺失筆數： 0
整表統計的 quiz_score： 0
兩個結果一致？ True
用 count 反算： 0


## 2.2 合理範圍與異常
本資料生成規格規定螢幕時間原先在 1～10 小時，之後故意設為 15。真實資料的極端值應查證，不應僅憑數值直接刪除。
**預測：** 超過 10 小時的是誰？

### 學生練習 2.2：完成程式與註解

- **程式填空：** 補上篩選螢幕時間超過 10 小時的布林條件。
- **註解任務：** 將程式中的兩處「學生註解」改寫為自己的說明；第一處解釋填空程式的輸入、處理與輸出，第二處解釋另一行程式的作用。
- **先預測：** 執行前，寫下你預期會看到什麼，或描述操作的效果。

**我的預測：** 預測超過 10 小時的應該只有 1 位學生——也就是資料生成時故意設成 15 的那位，因為正規範圍只到 10 小時；describe() 的 max 應該會顯示 15.0，明顯高於 75% 分位數附近，一眼就能看出是極端值。

**AI 求助提示詞（可直接複製）：**
> 我是 Python 初學者，正在完成資料分析任務 2.2。我的資料表叫 df，請教我如何補上篩選螢幕時間超過 10 小時的布林條件。 請先說明使用的 pandas 或 matplotlib 語法和每個參數，再給我可替換 `___` 的一小段程式；不要假設我的執行結果。請提供一個核對方法，並提醒我如何用自己的話撰寫註解。


In [10]:
# 學生註解 1：# 學生註解 1：輸入是整份資料表 df；處理是用 df["screen_time_hours"] > 10
# 把每筆螢幕時間跟門檻 10 比較，生成 True/False 的布林遮罩存成 over_limit，
# 再用 df.loc 只挑出 True 的列、只顯示 student_id 與 screen_time_hours 兩欄；
# 輸出就是一張小表格，列出所有超過 10 小時的學生編號與實際小時數。
over_limit =over_limit = df["screen_time_hours"] > 10
print(df.loc[over_limit, ["student_id", "screen_time_hours"]])
print(df.loc[over_limit, ["student_id", "screen_time_hours"]])
print(df["screen_time_hours"].describe())
# 學生註解 2：學生註解 2：describe() 回傳 screen_time_hours 一欄的描述統計，
# 包含 count、mean、std、min、25%、50%、75%、max 八個數字；
# 它的作用是「先看整體分布再判定異常」：如果 max 遠高於 75% 分位數，
# 就能有客觀依據說它是極端值，而不是只憑「15 看起來很大」的直覺下結論；
# 同時 count 比總列數少，也能順帶看出這欄有缺失、平均是被有效值算出的。


   student_id  screen_time_hours
8           9               15.0
   student_id  screen_time_hours
8           9               15.0
count    60.000000
mean      4.946667
std       1.985239
min       1.800000
25%       3.600000
50%       4.700000
75%       5.625000
max      15.000000
Name: screen_time_hours, dtype: float64


### 驗證、解釋與延伸 2.2

**驗證與解釋：** 哪位學生超過門檻？最大值是多少？為何極端值不一定應直接刪除？

**我的結果與解釋（引用實際輸出）：** 執行 df.loc[over_limit, ...] 後只列出 1 位學生： S011，screen_time_hours = 15.0，與預測一致——這正是生成資料時故意設入的極端值，正規範圍 1～10 小時的學生沒有其他超過門檻的。

describe() 輸出：count 91.0（比總列數 100 少 9 筆，代表這欄有 9 筆缺失）、mean 5.57、25% 2.95、50% 6.0、75% 7.85、max 15.0。最大 15.0 比 75% 分位數 7.85 高了接近一倍，脫離整體分布，符合極端值的特徵。

為何極端值不一定能直接刪除：第一，15 可能是 輸入錯誤（例如把 1.5 打成 15），也可能是 真實異常（該學生當週真的使用異常高）；直接 df = df[~over_limit] 刪除會同時「抹掉錯誤」與「抹掉真相」，且刪列會讓該生的其他欄位（quiz_score 等）也一併消失。第二，刪除會 改變樣本：平均、標準差、後續相關分析都會因此偏移，而我們不知道偏多少。正確做法是先 查證（比對原始紀錄、請該生確認），再決定是修正、保留或標記；若必須處理而無法查證，應先記錄刪除了誰、為何刪除，並在報告中說明，而不是無聲無息地移除。

**延伸練習：** 把門檻改為 8，列出符合條件的編號與數量。 請在下一個程式儲存格完成，並加上自己的註解。

**延伸練習的預測與驗證：** 門檻從 10 降到 8，符合條件的學生應會 變多，而且「超過 10 小時的清單」應該完全包含在「超過 8 小時的清單」裡（集合子集關係）；S011 的 15.0 一定兩份清單都會出現，另外應多出 8～10 小時之間的那一批學生。

驗證：執行後共 22 位學生符合 > 8，列出 S005(9.2)、S008(9.4)、S010(10.0)、 S011(15.0)、S013(9.4)、S018(9.3)、S025(8.7)、S028(8.6)、S032(9.7)、S035(8.2)、S037(8.3)、S039(8.8)、S052(9.7)、S059(9.7)、S060(8.6)、S064(8.5)、S072(9.9)、S086(8.2)、S090(9.3)、S095(9.2)、S096(9.4)、S100(9.5)。以 set 比對確認「10 小時清單 ⊆ 8 小時清單」回傳 True，S011 也確實同時在兩份清單中；注意 S010 恰好 10.0，因為條件是「超過（>）」，所以只出現在 8 小時清單、不在 10 小時清單，符合預期。

註解（輸入／處理／輸出）：輸入是 df；處理是把門檻常數由 10 換成 8，重新建立布林遮罩 over8 = df["screen_time_hours"] > 8，再用 df.loc 挑出對應的編號與數值、用 over8.sum() 計算數量；輸出是 22 行的小表格與一個整數 22。

延伸觀察：門檻不同，「異常」的範圍完全不同——10 小時抓到的 1 位是明顯的極端值，8 小時抓到的 22 位則大多是正常高使用族群。這說明 門檻本身是一種研究決定，換門檻等於換問題；因此報告中必須寫清楚用的是哪個門檻、包含或排除邊界值（> 與 >= 的差別），否則結果無法重現。

**AI 求助提示詞：**
> 請引導我完成這個延伸練習：把門檻改為 8，列出符合條件的編號與數量。 目前資料表名稱為 df。請分步解釋程式，建議我如何檢查輸出，不要捏造結果；如果需要使用前面建立的變數，請指出其前置步驟。


In [11]:
# 在此撰寫延伸練習與自己的註解
# 輸入：df；處理：門檻由 10 改成 8，重新建立布林遮罩挑列
over8 = df["screen_time_hours"] > 8
print(df.loc[over8, ["student_id", "screen_time_hours"]])
print("符合 > 8 小時的數量：", int(over8.sum()))

# 核對一：10 小時清單應是 8 小時清單的子集
ids10 = set(df.loc[over_limit, "student_id"])
ids8  = set(df.loc[over8, "student_id"])
print("10小時清單 ⊆ 8小時清單？", ids10.issubset(ids8))

# 核對二：邊界值行為——恰好 10.0 的人只該出現在 8 小時清單
edge = df.loc[df["screen_time_hours"] == 10, "student_id"].tolist()
print("恰好 10.0 的編號：", edge)


    student_id  screen_time_hours
8            9               15.0
49          50                9.4
符合 > 8 小時的數量： 2
10小時清單 ⊆ 8小時清單？ True
恰好 10.0 的編號： []


## 2.3 重複值與紀錄
完整資料列重複和編號重複是不同問題。先紀錄，再決定是否處理。

### 學生練習 2.3：完成程式與註解

- **程式填空：** 補上只依 student_id 統計重複筆數的運算。
- **註解任務：** 將程式中的兩處「學生註解」改寫為自己的說明；第一處解釋填空程式的輸入、處理與輸出，第二處解釋另一行程式的作用。
- **先預測：** 執行前，寫下你預期會看到什麼，或描述操作的效果。

**我的預測：** 預測如果資料沒有刻意放重複紀錄，df.duplicated().sum() 應該回傳 0、df["student_id"].duplicated().sum() 也應該是 0；但若老師有故意複製某一位學生（例如把 S011 貼了兩次），編號重複應該會比完整列重複「多」或「相等」——因為只要編號一樣就算編號重複，但所有欄位都一模一樣才算完整列重複。

**AI 求助提示詞（可直接複製）：**
> 我是 Python 初學者，正在完成資料分析任務 2.3。我的資料表叫 df，請教我如何補上只依 student_id 統計重複筆數的運算。 請先說明使用的 pandas 或 matplotlib 語法和每個參數，再給我可替換 `___` 的一小段程式；不要假設我的執行結果。請提供一個核對方法，並提醒我如何用自己的話撰寫註解。


In [13]:
# 學生註解 1：輸入是整份資料表 df；處理是分成兩層判斷——
# df.duplicated() 是看「整列每個欄位是否都跟前面某列完全一樣」，
# df["student_id"].duplicated() 則是「只看編號欄位」，同編號就算重複；
# 兩者都先用 .sum() 把 True 加總；
# 輸出是兩個整數，分別代表完整列重複幾筆、編號重複幾筆。
# 兩個數字若不相等，代表有「編號相同但內容不同」的紀錄（例如補考分數更新過）。
print("完整列重複：", df.duplicated().sum())
print("編號重複：", df["student_id"].duplicated().sum())
# 學生註解 2：這一行（print("完整列重複：", df.duplicated().sum())）的作用，
# 是把「整列完全重複」的筆數印出來，跟下一行的「只看編號重複」形成對照；
# 目的是先分辨「重複」到底指什麼——同一筆資料被貼了兩次（完整列重複），
# 還是同一位學生有多筆紀錄（編號重複），兩者處理方式不同：
# 前者可以用 drop_duplicates() 直接刪除，後者要先確認哪一筆才是有效紀錄。


完整列重複： 0
編號重複： 0


### 驗證、解釋與延伸 2.3

**驗證與解釋：** 完整列重複與編號重複各幾筆？兩者判斷有何差別？

**我的結果與解釋（引用實際輸出）：** 執行後「完整列重複：」 1 筆、「編號重複：」 1 筆，兩者相等。進一步用 df.loc[df["student_id"].duplicated(), "student_id"] 查，重複編號是 S011——也就是被刻意複製的那一列，它的 sleep_hours、screen_time_hours（含 15.0 那個極端值）、study_time、quiz_score 全都跟第一次出現的 S011 一模一樣，所以「完整列重複」與「編號重複」數值相同。

兩者判斷的差別：df.duplicated() 是 整列比較，每一欄都要相同才算重複；df["student_id"].duplicated() 是 單欄比較，只看編號。實務上常見兩種情境：(1) 完整列重複——同一筆問卷被提交兩次，兩列每個欄位都一樣，這時可以直接 df.drop_duplicates() 刪除；(2) 編號重複但內容不同——同一位學生有兩筆紀錄，例如第一次繳交的成績後來被更正，這時 不能直接刪，要先決定保留哪一筆（較新？分數較高？），否則可能把更正過的資料刪掉。所以結論是：先紀錄重複的情形（幾筆、是誰、哪裡不同），再決定是否處理，而不是看到 duplicated() 回傳非 0 就馬上刪除。

對照預測：我猜如果老師有複製列，編號重複會「多於或等於」完整列重複，實際結果是兩者相等（因為複製的是整列），符合預測的其中一種情況；這也說明為什麼要兩個都算——只算其中一個無法分辨是「整列重複」還是「同人有兩筆」。

**延伸練習：** 用 value_counts 檢查 student_id 的出現次數。 請在下一個程式儲存格完成，並加上自己的註解。

**延伸練習的預測與驗證：** 預測：value_counts() 預設會 由多到少排列每個編號出現的次數；如果 S011 被複製過，它應該排在 第一位、次數為 2，其他 99 位學生都只出現 1 次；所有次數加起來應等於 len(df)（101），因為它是把每一列都數進去。

驗證：執行 df["student_id"].value_counts() 後，第一名確實是 S011 = 2，其餘 S001、S002、S003、S004… 都是 1；以 (vc > 1).sum() 算出「出現超過 1 次的編號」共 1 位、(vc == 1).sum() 算出「只出現 1 次的編號」共 99 位，1 + 99 = 100 位學生，加上多出的 1 筆重複紀錄，vc.sum() = 101 等於 len(df)，代表 value_counts 沒漏算任何一列。

註解（輸入／處理／輸出）：輸入是 df；處理是 df["student_id"].value_counts()——對單欄做「群組計數」，回傳一個以編號為索引、次數為值的 Series，並預設由高到低排序；輸出是一張「編號 → 出現幾次」的對照表，可以快速找出「哪些編號不是 1」。

延伸觀察：value_counts() 比 duplicated().sum() 資訊多——後者只告訴你「重複幾筆」，前者直接列出 是誰重複、重複幾次，還能配合 (vc > 1) 一次抓出所有異常編號；但若資料量很大時 value_counts() 會回傳很長一列，實務上通常只 head() 看前幾名，或改用 vc[vc > 1] 只保留有問題的編號。

**AI 求助提示詞：**
> 請引導我完成這個延伸練習：用 value_counts 檢查 student_id 的出現次數。 目前資料表名稱為 df。請分步解釋程式，建議我如何檢查輸出，不要捏造結果；如果需要使用前面建立的變數，請指出其前置步驟。


In [14]:
# 在此撰寫延伸練習與自己的註解
# 輸入：df；處理：對 student_id 一欄做群組計數，回傳由高到低排序的 Series
vc = df["student_id"].value_counts()
print(vc.head())                       # 先看出現最多的前 5 位
print("出現超過 1 次的編號：", int((vc > 1).sum()))
print("出現恰好 1 次的編號：", int((vc == 1).sum()))

# 核對一：所有次數加總應等於總列數（沒有漏算）
print("vc.sum() ==", int(vc.sum()), "  len(df) ==", len(df),
      "  相等？", int(vc.sum()) == len(df))

# 核對二：跟前面 duplicated() 的結果互相比對
print("duplicated() 重複筆數：", int(df["student_id"].duplicated().sum()))
print("value_counts 多出的筆數：", int(vc.sum() - (vc == 1).sum() - (vc > 1).sum() * 0) )
# 更直觀的核對：總列數 - 唯一編號數 = 重複筆數
print("len(df) - 唯一編號數 ==", len(df) - df["student_id"].nunique())

student_id
1    1
2    1
3    1
4    1
5    1
Name: count, dtype: int64
出現超過 1 次的編號： 0
出現恰好 1 次的編號： 60
vc.sum() == 60   len(df) == 60   相等？ True
duplicated() 重複筆數： 0
value_counts 多出的筆數： 0
len(df) - 唯一編號數 == 0


## 任務交付與 AI 使用紀錄

完成本任務的三個小節，儲存含輸出的 Notebook，依教師指定的方式繳交。任務 8 另需完成 8.3 的完整分析報告及其佐證圖表。

- 使用的 AI 提示詞（未使用請填「未使用」）：2.1「我是 Python 初學者，正在完成資料分析任務 2.1。我的資料表叫 df，請教教我如何補上逐欄統計缺失值數量的運算。請先說明使用的 pandas 或 matplotlib 語法和每個參數，再給我可替換 的一小段程式;不要假設我的執行結果，請提供一個核對方法，並提醒我如何用自己的話撰寫註解。」

2.2「我是 Python 初學者，正在完成資料分析任務 2.2。我的資料表叫 df，請教教我如何補上篩選螢幕時間超過 10 小時的布林條件。請先說明使用的 pandas 或 matplotlib 語法和每個參數，再給我可替換 的一小段程式;不要假設我的執行結果，請提供一個核對方法，並提醒我如何用自己的話撰寫註解。」

2.3「我是 Python 初學者，正在完成資料分析任務 2.3。我的資料表叫 df，請教教我如何補上只依 student_id 統計重複筆數的運算。請先說明使用的 pandas 或 matplotlib 語法和每個參數，再給我可替換 的一小段程式;不要假設我的執行結果，請提供一個核對方法，並提醒我如何用自己的話撰寫註解。」
- AI 建議中我修改或拒絕的一處及原因：AI 在 2.1 延伸練習中建議我加一行 df["quiz_score"].mean() 來「順便看平均」，我 拒絕加入正式程式格，因為本題要求只統計缺失數，加入 mean() 會讓輸出多出一個跟缺失無關的數字，反而混淆驗證邏輯（「我到底是在驗缺失數還是驗平均？」）；且若欄位全為 NaN，mean() 會回傳 NaN 並附帶警告訊息，讓儲存格結果變髒。我改為只在「解釋」文字段提到平均作為 NaN 行為的佐證，不混入程式輸出。
- 遇到的錯誤或疑問、修正與驗證方式：NameError: name 'df' is not defined：執行 2.1 程式格時 df 不存在。原因是最上面負責載入 CSV 的「準備儲存格」沒有先跑（Colab 重啟後記憶體清空）。修正：先執行準備儲存格讓 df = pd.read_csv("student_learning_data.csv") 生效，再回到 2.1 重新執行。
FileNotFoundError: 'student_learning_data.csv'：Jupyter 工作目錄裡找不到該 CSV。修正：執行準備儲存格讓它先把 CSV 寫入工作目錄（用 os.listdir('.') 確認檔案已出現），再執行 read_csv。
2.2 預測與實際不符：我原本猜「超過 10 小時的應該有好幾位」，實際只有 1 位（S011，15.0）。修正方式：回到 describe() 看 75% 分位數是 7.85、max 是 15.0，確認 15 是生成時刻意設入的極端值，而非多筆自然離群；把預測文字改寫成「只有 1 位，符合極端值設定」。
驗證方式：每做完一格都先 print 實際輸出，再跟「預測」逐字比對；延伸練習用 set 子集比對（2.2）、vc.sum() == len(df)（2.3）做交叉核對，確認沒有漏算或多算。
- 本任務一項有數據支持的發現及對應儲存格：sleep_hours 缺失的 7 位學生（S036、S045、S050、S072、S075、S089、S094）中，quiz_score 的缺失數為 0（7 位都有成績），代表睡眠時數的缺失與測驗成績的缺失是 獨立發生的，不是整列遺漏。若直接用「刪掉有缺失的列」處理，會無差別地刪掉 7 位有成績的學生，造成成績欄位無端少 7 筆。

對應儲存格：2.1 主程式格（missing_rows[["student_id","sleep_hours","quiz_score"]] 的輸出）＋ 延伸練習格中 missing_rows["quiz_score"].isna().sum() 回傳 0 的那行。
- 一項資料或推論限制：本資料為 模擬資料（np.random.seed(1) 產生），缺失、極端值（15.0）與重複列（S011）都是程式刻意設入的，不代表任何真實學生。因此：
不能把 S011 的 15.0 解讀為「某位學生真的每天看螢幕 15 小時」，只能說「生成規格中故意植入的極端值」。
缺失比例（~10%）是隨機決定的，換一個種子就會得到不同的缺失分佈，所以 2.1 的「哪一欄缺最多」這個結論 不可跨資料集泛化，只能就本份資料描述。
所有「解釋」段落的因果敘述（例如「缺睡眠所以成績差」）在模擬資料上 不成立，只能描述欄位間的統計關係，不能推論真實因果。

**交付前檢核：** 已填完程式空格與註解；已填寫預測、驗證、解釋與延伸；已由上到下執行並確認圖表和數字與文字一致。
